# 05 Classical twin: training and cost

**Purpose.** Compare the complete hybrid pipeline with a *classical twin*: identical encoder, decoder, optimiser, schedule, validation-only checkpointing, explanation interface and audit chain; only the 8-qubit circuit is replaced by a dense 8x8 map with tanh (72 instead of 96 bottleneck parameters).

**What is stored.** Accuracy of the five re-trained twin models, and same-machine, single-thread timings of the simulated quantum model and the twin.
**Optional.** Set `RETRAIN = True` to retrain the twin (about 4 minutes per seed on one CPU thread; run the five seeds in parallel processes if you have the cores).

Absolute times depend on the machine; only the ratios between the two models are meaningful.

In [1]:
# --- setup: work from the laptop/ folder so that relative paths resolve ---
import os, sys, json
from pathlib import Path
ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
os.chdir(ROOT)
sys.path.insert(0, str(ROOT / "src"))
import numpy as np, pandas as pd
pd.set_option("display.precision", 4)
print("working folder:", ROOT)

working folder: <repo>\local_analysis


In [2]:
RETRAIN = False        # True: retrain the five twin models (a few minutes per seed)
RETIME  = False        # True: repeat the same-machine timing (needs the trained twin of seed 42)
import subprocess, glob
if RETRAIN:
    procs = [subprocess.Popen([sys.executable, "twin_pipeline.py", "multiclass", "train", str(s)]) for s in [42, 1, 7, 13, 2024]]
    [p.wait() for p in procs]
if RETIME:
    subprocess.run([sys.executable, "pqc_epoch_time.py"]); subprocess.run([sys.executable, "twin_pipeline.py", "multiclass", "time"])

In [3]:
rows = []
for f in sorted(glob.glob("results/twin_multiclass_seed*.json")):
    r = json.load(open(f))["seeds"]; s, v = list(r.items())[0]
    rows.append({"seed": int(s), "accuracy": v["acc"], "weighted F1": v["f1w"], "epochs run": v["epochs_run"]})
tw = pd.DataFrame(rows).set_index("seed"); display(tw)
print("mean accuracy %.4f, SD %.4f" % (tw["accuracy"].mean(), tw["accuracy"].std(ddof=1)))

,accuracy,weighted F1,epochs run
seed,,,
1,0.7768,0.7580,38
13,0.7916,0.7708,93
2024,0.7785,0.7582,48
42,0.7743,0.7571,32
7,0.7863,0.7680,111


mean accuracy 0.7815, SD 0.0072


In [4]:
T = json.load(open("results/twin_multiclass.json")); E = json.load(open("results/pqc_epoch_time.json"))
lat = T["latency_ms_per_sample"]; shp = T["kernelshap_forward_s_per_explanation"]
cost = pd.DataFrame([
  {"quantity": "parameters (total)", "quantum (simulated)": T["params"]["pqc"]["total"], "classical twin": T["params"]["twin"]["total"]},
  {"quantity": "training, s per epoch", "quantum (simulated)": E["pqc"]["epoch_s_extrapolated"], "classical twin": E["twin"]["epoch_s_extrapolated"]},
  {"quantity": "inference, ms/flow, batch 1", "quantum (simulated)": lat["1"]["pqc"], "classical twin": lat["1"]["twin"]},
  {"quantity": "inference, ms/flow, batch 128", "quantum (simulated)": lat["128"]["pqc"], "classical twin": lat["128"]["twin"]},
  {"quantity": "forward cost of one explanation, s", "quantum (simulated)": shp["pqc"], "classical twin": shp["twin"]}])
cost["ratio (quantum / twin)"] = cost["quantum (simulated)"] / cost["classical twin"]
cost

,quantity,quantum (simulated),classical twin,ratio (quantum / twin)
0,parameters (total),6768.0000,6744.0000,1.0036
1,"training, s per epoch",163.4825,4.2778,38.2169
2,"inference, ms/flow, batch 1",54.9263,0.4510,121.7743
3,"inference, ms/flow, batch 128",0.8590,0.0046,188.4418
4,"forward cost of one explanation, s",3.7704,0.0303,124.3520
